<a href="https://colab.research.google.com/github/Prajwala15/MLProject/blob/master/MLProject_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TextVQA — OCR-first vs Vision-Language Model vs Hybrid

Compares three approaches to text-based visual question answering:
- **OCR-first**: extract scene text with EasyOCR, then a text-only reader answers from it.
- **VLM**: a frozen BLIP-2 model answers directly from the image, no OCR.
- **Hybrid**: BLIP-2 answers the image with the OCR text as an added hint.

Key fixes baked into this notebook:
- One shared `LIMIT` drives every stage (OCR, VLM, OCR-first, hybrid), so all three are
  always scored on the exact same images.
- Uses the real ~25GB TextVQA image archive, not a small bundled demo subset.
- OCR results are cached and resumed, not recomputed from scratch on every rerun.
- VLM/hybrid inference is batched, not one image at a time.
- `TextVQADataset` filters to images that actually exist on disk before applying
  `limit`, so `limit=N` always means N usable samples.
- Every pipeline step streams its output live and fails loudly on error.

## Setup

In [ ]:
%cd /content
!rm -rf textvqa
!git clone https://github.com/Prajwala15/MLProject.git textvqa
%cd textvqa
!ls

/content
Cloning into 'textvqa'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 85 (delta 36), reused 46 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (85/85), 2.23 MiB | 11.74 MiB/s, done.
Resolving deltas: 100% (36/36), done.
Filtering content: 100% (2/2), 265.46 MiB | 12.54 MiB/s, done.
/content/textvqa
configs			      outputs		scripts
data			      README.md		src
MLProject_with_Member3.ipynb  requirements.txt	updatedCode.ipynb


In [ ]:
!pip install -r requirements.txt
!pip install easyocr pytesseract opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.2/296.2 kB 17.2 MB/s eta 0:00:00


Hugging Face token

`blip2-flan-t5-xl` is a large download (~15GB). Unauthenticated HF requests get
rate-limited, which can make the VLM step look "stuck." Get a free read-only token at
https://huggingface.co/settings/tokens and paste it below (safe to skip).

In [ ]:
import os
HF_TOKEN = ""  # paste your token here, e.g. "hf_xxxxxxxxxxxxxxxxxxxxxxxx"
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN set")
else:
    print("No HF_TOKEN set — downloads may be slower/rate-limited but should still work")

No HF_TOKEN set — downloads may be slower/rate-limited but should still work


### Shared config

Set `LIMIT` once. `0` = full val split; a smaller number (e.g. 200) is a fast smoke
test. Every stage below uses this same value, so the comparison is always apples-to-apples.

In [ ]:
LIMIT = 200  # 0 = full val split. Keep this the SAME everywhere below.

### Data: annotations + the real image archive

`train_val_images.zip` is the official ~25GB TextVQA/OpenImages archive. It unzips to
`train_images/`, matching `configs/default.yaml`'s default path. This step is slow —
expect anywhere from ~5 to 30+ minutes depending on Colab's network that session.

In [ ]:
%cd /content/textvqa/data
!wget https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_train.json
!wget https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_val.json
!ls -lh *.json

/content/textvqa/data
--2026-07-09 21:35:57--  https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_train.json
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 54.230.79.53, 54.230.79.88, 54.230.79.93, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|54.230.79.53|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 21634937 (21M) [text/plain]
Saving to: ‘TextVQA_0.5.1_train.json’

TextVQA_0.5.1_train 100%[===================>]  20.63M  --.-KB/s    in 0.1s    

2026-07-09 21:35:58 (165 MB/s) - ‘TextVQA_0.5.1_train.json’ saved [21634937/21634937]

--2026-07-09 21:35:58--  https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_val.json
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 54.230.79.53, 54.230.79.88, 54.230.79.93, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|54.230.79.53|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3116162 (3.0M) [text/plain]
Saving to: ‘TextV

In [ ]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   48G   66G  42% /


In [ ]:
%cd /content/textvqa/data
!rm -f train_val_images.zip train_val_images.zip.*
!wget -q --show-progress https://dl.fbaipublicfiles.com/textvqa/images/train_val_images.zip
!unzip -q train_val_images.zip
!rm -f train_val_images.zip
!ls -d train_images && echo "OK: train_images/ created"
%cd /content/textvqa

/content/textvqa/data
train_val_images.zi 100%[===================>]   6.59G   224MB/s    in 46s     
train_images
OK: train_images/ created
/content/textvqa


In [ ]:
!grep -A5 "images:" configs/default.yaml

  images:
    train: data/train_images
    val:   data/train_images   # val images live in the train_images folder for TextVQA 0.5.1
    test:  data/test_images
  ocr_cache: outputs/ocr_cache   # one json per split
  preds:     outputs/preds


### One-time patches

1. `src/ocr.py` / `scripts/01_run_ocr.py`: numpy floats/ints aren't JSON-serializable —
   cast to native Python types before dumping.
2. `src/dataset.py`: filter to images that exist on disk **before** applying `limit`, so
   `limit=N` always yields N usable samples.
3. `scripts/02_eval.py`: batch VLM/hybrid inference via `answer_batch()` instead of one
   image at a time.

In [ ]:
import os
os.chdir("/content/textvqa")

# fix A: bbox -> float in src/ocr.py (numpy floats aren't JSON-serializable)
op = "src/ocr.py"
s = open(op).read()
if "float(min(xs))" not in s:
    s = s.replace('"bbox": [min(xs), min(ys), max(xs), max(ys)],',
                  '"bbox": [float(min(xs)), float(min(ys)), float(max(xs)), float(max(ys))],')
    open(op, "w").write(s)
print("ocr.py patched:", "float(min(xs))" in open(op).read())

# fix B: numpy-safe json.dump in scripts/01_run_ocr.py
rp = "scripts/01_run_ocr.py"
r = open(rp).read()
if "_to_native" not in r:
    helper = ("\ndef _to_native(o):\n"
              "    import numpy as np\n"
              "    if isinstance(o, np.integer): return int(o)\n"
              "    if isinstance(o, np.floating): return float(o)\n"
              "    if isinstance(o, np.ndarray): return o.tolist()\n"
              "    raise TypeError(f'not serializable: {type(o)}')\n\n")
    lines = r.splitlines(keepends=True)
    idx = max(i for i, l in enumerate(lines) if l.startswith(("import ", "from ")))
    lines.insert(idx + 1, helper)
    r = "".join(lines)
r = r.replace("json.dump(cache, f)", "json.dump(cache, f, default=_to_native)")
open(rp, "w").write(r)
print("run_ocr.py patched:", open(rp).read().count("default=_to_native"))

# fix C: TextVQADataset silently sliced records[:limit] with no check that the
# image file exists -> crashes downstream (VLM/hybrid) with FileNotFoundError if any
# of the first N records has no image on disk. Filter to existing images BEFORE
# applying limit, so `limit` always means "N usable samples".
dsp = "src/dataset.py"
ds = open(dsp).read()
old_init = (
    '        with open(ann_path) as f:\n'
    '            blob = json.load(f)\n'
    '        self.records = blob["data"] if isinstance(blob, dict) and "data" in blob else blob\n'
    '        if limit:\n'
    '            self.records = self.records[:limit]'
)
new_init = (
    '        with open(ann_path) as f:\n'
    '            blob = json.load(f)\n'
    '        records = blob["data"] if isinstance(blob, dict) and "data" in blob else blob\n'
    '\n'
    '        # Keep only records whose image actually exists on disk, THEN apply limit --\n'
    '        # otherwise `limit` can select records with no image file and every\n'
    '        # downstream stage crashes.\n'
    '        kept = []\n'
    '        for r in records:\n'
    '            p = _resolve_image_path(self.image_dir, r["image_id"])\n'
    '            if os.path.exists(p):\n'
    '                kept.append(r)\n'
    '                if limit and len(kept) >= limit:\n'
    '                    break\n'
    '        self.records = kept'
)
if old_init in ds:
    ds = ds.replace(old_init, new_init)
    open(dsp, "w").write(ds)
print("dataset.py patched to skip missing images:", "Keep only records" in open(dsp).read())

ocr.py patched: True
run_ocr.py patched: 0
dataset.py patched to skip missing images: True


In [ ]:
%%writefile scripts/02_eval.py
#!/usr/bin/env python
"""Run one pipeline over a split, score it, and dump per-sample predictions.

Usage:
    python scripts/02_eval.py --approach ocr_first --split val [--limit 100]
    python scripts/02_eval.py --approach vlm       --split val
    python scripts/02_eval.py --approach hybrid    --split val

Predictions are written to outputs/preds/<approach>_<split>.json including the OCR
coverage bucket per sample, which 03_analysis.py consumes.

Batches vlm / hybrid inference via answer_batch() instead of one image at a time.
"""
import argparse
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from tqdm import tqdm

from src.config import load_config, pick_device
from src.dataset import TextVQADataset
from src.metrics import aggregate, anls, vqa_accuracy
from src.ocr import tokens_to_string
from src.ocr_quality import coverage_bucket


def load_ocr_cache(cfg, split):
    engine = cfg["ocr"]["engine"]
    path = os.path.join(cfg["paths"]["ocr_cache"], f"{split}_{engine}.json")
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"OCR cache missing: {path}. Run scripts/01_run_ocr.py first.")
    with open(path) as f:
        return json.load(f)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--approach", required=True,
                    choices=["ocr_first", "vlm", "hybrid"])
    ap.add_argument("--split", default="val")
    ap.add_argument("--limit", type=int, default=None)
    ap.add_argument("--batch_size", type=int, default=None)
    args = ap.parse_args()

    cfg = load_config()
    device = pick_device(cfg["eval"]["device"])
    limit = args.limit if args.limit is not None else cfg["eval"]["limit"]
    batch_size = args.batch_size or cfg["eval"].get("batch_size", 8)
    ds = TextVQADataset(args.split, cfg, limit=limit)

    need_ocr = args.approach in ("ocr_first", "hybrid")
    ocr_cache = load_ocr_cache(cfg, args.split) if need_ocr else {}
    mt, mc = cfg["ocr"]["max_tokens"], cfg["ocr"]["min_confidence"]

    if args.approach == "ocr_first":
        from src.models.ocr_first import build_ocr_first
        model = build_ocr_first(cfg, device)
    elif args.approach == "vlm":
        from src.models.vlm import build_vlm
        model = build_vlm(cfg, device)
    else:
        from src.models.hybrid import build_hybrid
        model = build_hybrid(cfg, device)

    n = len(ds)
    metas = [ds.meta(i) for i in range(n)]
    ocr_texts = [
        tokens_to_string(ocr_cache.get(str(m["image_id"]), []), mt, mc) if need_ocr else ""
        for m in metas
    ]
    preds = [None] * n

    if args.approach == "ocr_first":
        for i in tqdm(range(n), desc="eval[ocr_first]"):
            preds[i] = model.answer(metas[i]["question"], ocr_texts[i])
    else:
        for start in tqdm(range(0, n, batch_size), desc=f"eval[{args.approach}]"):
            idxs = list(range(start, min(start + batch_size, n)))
            images = [ds[i]["image"] for i in idxs]
            questions = [metas[i]["question"] for i in idxs]
            if args.approach == "vlm":
                batch_preds = model.answer_batch(images, questions)
            else:
                hint_qs = []
                for i in idxs:
                    ocr_hint = ocr_texts[i] if ocr_texts[i] else "(none)"
                    hint_qs.append(
                        "Text spotted in the image (may be noisy): " + ocr_hint +
                        ". Question: " + metas[i]["question"]
                    )
                batch_preds = model.vlm.answer_batch(images, hint_qs)
            for j, i in enumerate(idxs):
                preds[i] = batch_preds[j]

    rows = []
    for i, m in enumerate(metas):
        ocr_text = ocr_texts[i]
        bucket = coverage_bucket(ocr_text, m["answers"]) if m["answers"] else "unknown"
        rows.append({
            "question_id": m["question_id"],
            "question": m["question"],
            "pred": preds[i],
            "answers": m["answers"],
            "ocr_text": ocr_text,
            "ocr_bucket": bucket,
            "vqa_acc": vqa_accuracy(preds[i], m["answers"]),
            "anls": anls(preds[i], m["answers"]),
        })

    os.makedirs(cfg["paths"]["preds"], exist_ok=True)
    os.makedirs(cfg["paths"]["scores"], exist_ok=True)
    pred_path = os.path.join(cfg["paths"]["preds"], f"{args.approach}_{args.split}.json")
    with open(pred_path, "w") as f:
        json.dump(rows, f, indent=2)

    summary = aggregate(rows)
    summary["approach"] = args.approach
    summary["split"] = args.split
    score_path = os.path.join(cfg["paths"]["scores"], f"{args.approach}_{args.split}.json")
    with open(score_path, "w") as f:
        json.dump(summary, f, indent=2)

    print(json.dumps(summary, indent=2))
    print(f"\npredictions -> {pred_path}")


if __name__ == "__main__":
    main()

Overwriting scripts/02_eval.py


### `run_step`: stream pipeline scripts live, fail loudly

Prints output as it happens (so `tqdm` progress bars are visible) and raises
immediately with a clear error if a script fails.

In [ ]:
import subprocess, sys, os

def run_step(cmd, cwd="/content/textvqa"):
    print(">", " ".join(cmd))
    process = subprocess.Popen(
        cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="")
    process.wait()
    if process.returncode != 0:
        raise RuntimeError(f"Step failed (exit {process.returncode}): {' '.join(cmd)}")
    return process.returncode

### Sanity check: dataset

In [ ]:
import sys
sys.path.insert(0, "/content/textvqa")
os.chdir("/content/textvqa")

from src.dataset import TextVQADataset
d = TextVQADataset("val", limit=LIMIT)
print("val samples (LIMIT applied, missing-image records already filtered out):", len(d))
print("example:", d[0]["question"], "->", d[0]["answers"][:3])

val samples (LIMIT applied, missing-image records already filtered out): 200
example: what is the brand of this camera? -> ['nous les gosses', 'dakota', 'clos culombu']


In [ ]:
import torch
print("GPU:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY — set Runtime->GPU")

GPU: True | Tesla T4


---
## OCR

Extract scene text with EasyOCR (cached — safe to rerun, it resumes instead of restarting).

In [ ]:
run_step(["python", "scripts/01_run_ocr.py", "--split", "val",
          "--engine", "easyocr", "--gpu", "--limit", str(LIMIT)])

> python scripts/01_run_ocr.py --split val --engine easyocr --gpu --limit 200


**View OCR output**

In [ ]:
import json

with open("outputs/ocr_cache/val_easyocr.json") as f:
    data = json.load(f)

for img_id, tokens in data.items():
    if len(tokens) > 0:
        print("Image ID:", img_id)
        print(tokens[:5])
        break

print("Successfully loaded val_easyocr.json! images cached:", len(data))

**OCR confidence overview**

In [ ]:
import numpy as np

conf = [t["conf"] for tokens in data.values() for t in tokens]
print("Total OCR tokens:", len(conf))
print("Average confidence:", np.mean(conf))
print("Min confidence:", np.min(conf))
print("Max confidence:", np.max(conf))

---
## Vision-Language Model (VLM) approach

Frozen BLIP-2 answers questions directly from the image (zero-shot, no OCR, no training).
Batched inference over the same `LIMIT` images the OCR stage covered.

In [ ]:
run_step(["python", "scripts/02_eval.py", "--approach", "vlm",
          "--split", "val", "--limit", str(LIMIT)])

**Qualitative examples: cases BLIP-2 reads correctly without OCR**

In [ ]:
import json
from IPython.display import display
from src.normalize import normalize_answer

preds = json.load(open("outputs/preds/vlm_val.json"))
idx_by_qid = {d.meta(i)["question_id"]: i for i in range(len(d))}

good = [p for p in preds if p.get("vqa_acc", 0) >= 0.6
        and normalize_answer(p["pred"]) not in {"yes", "no", "0", "1", "2"}
        and len(normalize_answer(p["pred"])) >= 2]

print(f"{len(good)} correct 'reading' cases. Showing up to 6:\n")
for p in good[:6]:
    i = idx_by_qid.get(p["question_id"])
    if i is None:
        continue
    s = d[i]
    im = s["image"]; w = 360
    display(im.resize((w, int(im.height * w / im.width))))
    print("Q       :", p["question"])
    print("VLM pred:", p["pred"], " (correct — read from image, no OCR)")
    print("GT      :", p["answers"][:3])
    print("-" * 50)

---
## OCR-first and Hybrid approaches, comparison, and failure cases

OCR-first uses the repo's BLIP-2/Flan-T5 text reader over OCR text. Hybrid runs the VLM
with the OCR text as an added hint. Both scored on the same `LIMIT` images as the VLM run above.

In [ ]:
run_step(["python", "scripts/02_eval.py", "--approach", "ocr_first",
          "--split", "val", "--limit", str(LIMIT)])

In [ ]:
run_step(["python", "scripts/02_eval.py", "--approach", "hybrid",
          "--split", "val", "--limit", str(LIMIT)])

**Final comparison: overall + accuracy by OCR-coverage bucket**

In [ ]:
run_step(["python", "scripts/03_analysis.py", "--split", "val"])

In [ ]:
import pandas as pd
from IPython.display import Image as IPImage, display

table = pd.read_csv("outputs/comparison_val.csv", index_col=0)
display(table.round(3))
display(IPImage("outputs/comparison_val.png"))

**Failure analysis: OCR misreads vs VLM hallucinations**

In [ ]:
import json
from IPython.display import display

ocr_rows = {r["question_id"]: r for r in json.load(open("outputs/preds/ocr_first_val.json"))}
vlm_rows = {r["question_id"]: r for r in json.load(open("outputs/preds/vlm_val.json"))}
idx_by_qid = {d.meta(i)["question_id"]: i for i in range(len(d))}

def show(qid, label):
    if qid not in idx_by_qid:
        return
    s = d[idx_by_qid[qid]]
    im = s["image"]; w = 320
    display(im.resize((w, int(im.height * w / im.width))))
    o, v = ocr_rows[qid], vlm_rows[qid]
    print("CASE:", label)
    print("Q        :", o["question"])
    print("OCR text :", (o["ocr_text"][:70] or "(none)"))
    print("OCR-first:", o["pred"] or "(blank)", f"[acc {o['vqa_acc']:.1f}]")
    print("VLM      :", v["pred"], f"[acc {v['vqa_acc']:.1f}]")
    print("GT       :", o["answers"][:3])
    print("=" * 55)

print("\n### VLM succeeds where OCR-first fails ###")
shown = 0
for qid, o in ocr_rows.items():
    v = vlm_rows.get(qid)
    if v and o["vqa_acc"] < 0.3 and v["vqa_acc"] > 0.7:
        show(qid, "OCR failed -> VLM read it"); shown += 1
    if shown >= 2:
        break

print("\n### OCR-first succeeds where VLM hallucinates ###")
shown = 0
for qid, o in ocr_rows.items():
    v = vlm_rows.get(qid)
    if v and o["vqa_acc"] > 0.7 and v["vqa_acc"] < 0.3:
        show(qid, "VLM hallucinated -> OCR correct"); shown += 1
    if shown >= 2:
        break

## Save deliverables

In [ ]:
from google.colab import files
import os

deliverables = [
    "outputs/comparison_val.csv",
    "outputs/comparison_val.png",
    "outputs/preds/ocr_first_val.json",
    "outputs/preds/vlm_val.json",
    "outputs/preds/hybrid_val.json",
    "outputs/scores/ocr_first_val.json",
    "outputs/scores/vlm_val.json",
    "outputs/scores/hybrid_val.json",
]
for f in deliverables:
    if os.path.exists(f):
        files.download(f)
print("Deliverables saved")

# **GPU Restart**

In [ ]:
import json, os
import pandas as pd

scores = {}
for approach in ["ocr_first", "vlm", "hybrid"]:
    path = f"outputs/scores/{approach}_val.json"
    if os.path.exists(path):
        with open(path) as f:
            scores[approach] = json.load(f)

summary = pd.DataFrame(scores).T[["n", "vqa_accuracy", "anls"]]
summary["vqa_accuracy"] = (summary["vqa_accuracy"] * 100).round(1).astype(str) + "%"
summary["anls"] = summary["anls"].round(3)
print(summary)

In [ ]:
import gc, torch

for name in ["vlm_model", "ocr_first_model", "hybrid_model", "model", "ocr_engine"]:
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()
print(f"free GPU memory: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

In [ ]:
from google.colab import files
from PIL import Image
from IPython.display import display
import torch

from src.config import load_config, pick_device
from src.ocr import build_engine, tokens_to_string
from src.models.vlm import build_vlm

if "vlm_model" not in globals():
    cfg = load_config()
    device = pick_device(cfg["eval"]["device"])
    print("loading OCR engine + BLIP-2 (one copy, reused for all three approaches)...")
    ocr_engine = build_engine(cfg["ocr"]["engine"], tuple(cfg["ocr"]["langs"]),
                               gpu=torch.cuda.is_available())
    vlm_model = build_vlm(cfg, device)          # the only large model we load
    print("loaded\n")

def ocr_first_answer(question, ocr_text, max_new_tokens=16):
    """Reuses vlm_model's own language-model weights as a text-only reader —
    avoids loading a second full flan-t5-xl checkpoint just for this."""
    prompt = (f"Read the text found in an image and answer the question with the "
              f"shortest possible phrase (one word or number when possible).\n"
              f"Text in image: {ocr_text or '(none)'}\nQuestion: {question}\nAnswer:")
    tok = vlm_model.proc.tokenizer(prompt, return_tensors="pt",
                                    truncation=True, max_length=512).to(vlm_model.device)
    with torch.no_grad():
        out = vlm_model.model.language_model.generate(**tok, max_new_tokens=max_new_tokens)
    return vlm_model.proc.tokenizer.decode(out[0], skip_special_tokens=True).strip()

def hybrid_answer(image, question, ocr_text):
    """Same idea: reuse vlm_model instead of a second BLIP2VLM copy."""
    hint_q = f"Text spotted in the image (may be noisy): {ocr_text or '(none)'}. Question: {question}"
    return vlm_model.answer(image, hint_q)

# --- upload ---
print("upload an image:")
uploaded = files.upload()
fname = list(uploaded.keys())[-1]
image = Image.open(fname).convert("RGB")
display(image.resize((360, int(image.height * 360 / image.width))))

tokens = ocr_engine.read(image)
ocr_text = tokens_to_string(tokens, cfg["ocr"]["max_tokens"], cfg["ocr"]["min_confidence"])
print("OCR text found:", ocr_text or "(none)")

# --- ask as many questions as you want; type 'done' to stop ---
while True:
    question = input("\nAsk a question (or type 'done'): ").strip()
    if question.lower() in ("done", "exit", "quit", ""):
        break
    print("VLM      :", vlm_model.answer(image, question))
    print("OCR-first:", ocr_first_answer(question, ocr_text))
    print("Hybrid   :", hybrid_answer(image, question, ocr_text))